# Modeles de Base — Classification binaire de Fake News
## Pipeline : TF-IDF / GloVe + Classifieurs (LogReg, SVC, XGBoost)

**Objectif** : Construire des baselines solides pour la detection de fake news en classification binaire (Fake/Real).

**Approches** :
1. **TF-IDF + LogisticRegression** (avec et sans SMOTE)
2. **TF-IDF + LinearSVC** (avec et sans SMOTE)
3. **TF-IDF + XGBoost** (gradient boosting sur features sparse)
4. **Word2Vec / GloVe Embeddings + LogisticRegression**
5. Comparaison des performances et selection du meilleur modele

**Pre-requis** : Avoir execute le notebook `EDA_LIAR.ipynb` pour generer les fichiers Parquet.

In [15]:
import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings("ignore")

# Chemins
DATA_DIR = Path("../data/Traitees")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Imports OK")

Imports OK


## 1. Chargement des donnees preprocessees

In [16]:
df_train = pd.read_parquet(DATA_DIR / "liar_train.parquet")
df_valid = pd.read_parquet(DATA_DIR / "liar_valid.parquet")
df_test  = pd.read_parquet(DATA_DIR / "liar_test.parquet")

# Features et targets
X_train = df_train["statement_clean"]
y_train = df_train["label_binary"]

X_valid = df_valid["statement_clean"]
y_valid = df_valid["label_binary"]

X_test = df_test["statement_clean"]
y_test = df_test["label_binary"]

print(f"Train : {len(X_train):,} | Valid : {len(X_valid):,} | Test : {len(X_test):,}")
print(f"\nDistribution train : {y_train.value_counts().to_dict()}")
print(f"Distribution test  : {y_test.value_counts().to_dict()}")

Train : 10,240 | Valid : 1,284 | Test : 1,267

Distribution train : {1: 5752, 0: 4488}
Distribution test  : {1: 714, 0: 553}


## 2. Feature Engineering : TF-IDF

**TF-IDF** (Term Frequency - Inverse Document Frequency) transforme le texte en vecteurs numeriques.
- **Avantages** : Simple, rapide, efficace pour la classification de texte court
- **Limites** : Ne capture ni le sens semantique, ni l'ordre des mots

On explore les hyperparametres : `ngram_range`, `min_df`, `max_df`, `sublinear_tf`.

In [17]:
# Exploration du TF-IDF : taille du vocabulaire selon les parametres
tfidf_test = TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2), min_df=2, max_df=0.9)
X_tfidf = tfidf_test.fit_transform(X_train)

print(f"Matrice TF-IDF : {X_tfidf.shape}")
print(f"Vocabulaire : {len(tfidf_test.vocabulary_):,} features")
print(f"Densite moyenne : {X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1]) * 100:.3f}%")

Matrice TF-IDF : (10240, 24814)
Vocabulaire : 24,814 features
Densite moyenne : 0.101%


## 3. Entrainement des modeles de base (sans SMOTE)

### 3.1 TF-IDF + LogisticRegression

La regression logistique est un choix classique pour la classification de texte. On utilise `class_weight='balanced'` pour compenser le desequilibre des classes et `GridSearchCV` pour optimiser les hyperparametres.

In [18]:
# Grille d'hyperparametres commune
param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [2, 5],
    "tfidf__max_df": [0.9, 1.0],
    "clf__C": [0.1, 1.0, 10.0],
}

# Dictionnaire pour stocker tous les resultats
all_results = {}

# --- LogisticRegression ---
pipe_lr = Pipeline([
    ("tfidf", TfidfVectorizer(sublinear_tf=True)),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])

print("[LogisticRegression] GridSearchCV (3-fold)...")
grid_lr = GridSearchCV(pipe_lr, param_grid, cv=3, scoring="f1_weighted", n_jobs=-1, verbose=1)
grid_lr.fit(X_train, y_train)

best_lr = grid_lr.best_estimator_
print(f"\nMeilleurs params : {grid_lr.best_params_}")
print(f"Meilleur F1 CV   : {grid_lr.best_score_:.4f}")

[LogisticRegression] GridSearchCV (3-fold)...
Fitting 3 folds for each of 24 candidates, totalling 72 fits

Meilleurs params : {'clf__C': 0.1, 'tfidf__max_df': 0.9, 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 2)}
Meilleur F1 CV   : 0.6144


In [19]:
# Evaluation LogisticRegression
def evaluate_and_store(model, name, X_tr, y_tr, X_te, y_te, results_dict):
    """Evalue un modele et stocke les resultats."""
    y_pred_tr = model.predict(X_tr)
    y_pred_te = model.predict(X_te)
    
    acc_tr = accuracy_score(y_tr, y_pred_tr)
    f1_tr = f1_score(y_tr, y_pred_tr, average="weighted")
    acc_te = accuracy_score(y_te, y_pred_te)
    f1_te = f1_score(y_te, y_pred_te, average="weighted")
    prec_te = precision_score(y_te, y_pred_te, average="weighted")
    rec_te = recall_score(y_te, y_pred_te, average="weighted")
    
    metrics = {
        "train_acc": round(acc_tr, 4), "train_f1": round(f1_tr, 4),
        "test_acc": round(acc_te, 4), "test_f1": round(f1_te, 4),
        "test_precision": round(prec_te, 4), "test_recall": round(rec_te, 4),
        "gap_acc": round(acc_tr - acc_te, 4),
        "cm": confusion_matrix(y_te, y_pred_te).tolist(),
    }
    results_dict[name] = metrics
    
    print(f"\n=== {name} ===")
    print(f"Train — Acc: {acc_tr:.4f}  F1: {f1_tr:.4f}")
    print(f"Test  — Acc: {acc_te:.4f}  F1: {f1_te:.4f}  P: {prec_te:.4f}  R: {rec_te:.4f}")
    print(f"Gap Acc: {acc_tr - acc_te:.4f}")
    print(f"\n{classification_report(y_te, y_pred_te, target_names=['Fake', 'Real'])}")
    
    return metrics

metrics_lr = evaluate_and_store(best_lr, "TF-IDF + LogReg", X_train, y_train, X_test, y_test, all_results)


=== TF-IDF + LogReg ===
Train — Acc: 0.6832  F1: 0.6844
Test  — Acc: 0.6006  F1: 0.6022  P: 0.6094  R: 0.6006
Gap Acc: 0.0826

              precision    recall  f1-score   support

        Fake       0.54      0.62      0.58       553
        Real       0.67      0.59      0.62       714

    accuracy                           0.60      1267
   macro avg       0.60      0.60      0.60      1267
weighted avg       0.61      0.60      0.60      1267



### 3.2 TF-IDF + LinearSVC

Le SVM lineaire est souvent performant sur des donnees a haute dimension (sparse TF-IDF). On utilise aussi `class_weight='balanced'`.

In [20]:
# --- LinearSVC ---
pipe_svc = Pipeline([
    ("tfidf", TfidfVectorizer(sublinear_tf=True)),
    ("clf", LinearSVC(class_weight="balanced", max_iter=2000, random_state=42)),
])

print("[LinearSVC] GridSearchCV (3-fold)...")
grid_svc = GridSearchCV(pipe_svc, param_grid, cv=3, scoring="f1_weighted", n_jobs=-1, verbose=1)
grid_svc.fit(X_train, y_train)

best_svc = grid_svc.best_estimator_
print(f"\nMeilleurs params : {grid_svc.best_params_}")
print(f"Meilleur F1 CV   : {grid_svc.best_score_:.4f}")

metrics_svc = evaluate_and_store(best_svc, "TF-IDF + LinearSVC", X_train, y_train, X_test, y_test, all_results)

[LinearSVC] GridSearchCV (3-fold)...
Fitting 3 folds for each of 24 candidates, totalling 72 fits

Meilleurs params : {'clf__C': 0.1, 'tfidf__max_df': 0.9, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2)}
Meilleur F1 CV   : 0.6158

=== TF-IDF + LinearSVC ===
Train — Acc: 0.8270  F1: 0.8274
Test  — Acc: 0.6401  F1: 0.6407  P: 0.6417  R: 0.6401
Gap Acc: 0.1869

              precision    recall  f1-score   support

        Fake       0.58      0.61      0.60       553
        Real       0.69      0.67      0.68       714

    accuracy                           0.64      1267
   macro avg       0.64      0.64      0.64      1267
weighted avg       0.64      0.64      0.64      1267



### 3.3 TF-IDF + XGBoost

**XGBoost** (eXtreme Gradient Boosting) est un algorithme de boosting par arbres de decision. Contrairement aux modeles lineaires (LogReg, SVC), il peut capturer des **interactions non lineaires** entre les features TF-IDF.

**Avantages** :
- Robuste a l'overfitting grace a la regularisation (max_depth, subsample, min_child_weight)
- Gere bien les features sparse (TF-IDF)
- Souvent competitif ou superieur aux modeles lineaires sur les datasets tabulaires

On utilise le meme pipeline TF-IDF optimise, puis on entraine XGBoost avec GridSearchCV.

In [31]:
from xgboost import XGBClassifier
from scipy.sparse import hstack, csr_matrix

# === TF-IDF + XGBoost enrichi ===
# Combine texte (statement + subject + context) + metadata speaker
# Entraine sur train + valid pour maximiser les donnees

df_tv = pd.concat([df_train, df_valid], ignore_index=True)
y_tv = df_tv["label_binary"]

# Speaker credibility (target encoding sur train+valid)
speaker_cred = df_tv.groupby("speaker")["label_binary"].mean().to_dict()
speaker_count = df_tv.groupby("speaker")["label_binary"].count().to_dict()
global_mean = y_tv.mean()

def build_meta(df):
    return csr_matrix(np.column_stack([
        df["credibility_score"].values,
        df["speaker"].map(speaker_cred).fillna(global_mean).values,
        np.log1p(df["speaker"].map(speaker_count).fillna(0).values),
        df["n_words"].values,
    ]))

# TF-IDF sur texte enrichi (statement + subject + context)
df_tv["full_text"] = (df_tv["statement_nlp"].fillna("") + " " +
                       df_tv["subject"].fillna("") + " " +
                       df_tv["context"].fillna(""))
df_test["full_text"] = (df_test["statement_nlp"].fillna("") + " " +
                         df_test["subject"].fillna("") + " " +
                         df_test["context"].fillna(""))

tfidf_xgb = TfidfVectorizer(
    max_features=5000, ngram_range=(1, 2),
    min_df=2, max_df=0.95, sublinear_tf=True
)
X_tr_tfidf = tfidf_xgb.fit_transform(df_tv["full_text"])
X_te_tfidf = tfidf_xgb.transform(df_test["full_text"])

X_tr_combined = hstack([X_tr_tfidf, build_meta(df_tv)])
X_te_combined = hstack([X_te_tfidf, build_meta(df_test)])
print(f"Features : {X_tr_combined.shape[1]} (TF-IDF: {X_tr_tfidf.shape[1]} + meta: 4)")

# XGBoost fortement regularise (eviter overfitting)
xgb_model = XGBClassifier(
    n_estimators=1000,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.3,
    min_child_weight=10,
    reg_alpha=1.0,
    reg_lambda=5.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

print("[XGBoost enrichi] Entrainement (train+valid)...")
xgb_model.fit(X_tr_combined, y_tv)

# Evaluation sur test
y_pred_tr = xgb_model.predict(X_tr_combined)
y_pred_te = xgb_model.predict(X_te_combined)

acc_tr = accuracy_score(y_tv, y_pred_tr)
f1_tr = f1_score(y_tv, y_pred_tr, average="weighted")
acc_te = accuracy_score(y_test, y_pred_te)
f1_te = f1_score(y_test, y_pred_te, average="weighted")
p_te = precision_score(y_test, y_pred_te, average="weighted")
r_te = recall_score(y_test, y_pred_te, average="weighted")

print(f"=== TF-IDF + XGBoost (enrichi) ===")
print(f"Train -> Acc: {acc_tr:.4f}  F1: {f1_tr:.4f}")
print(f"Test  -> Acc: {acc_te:.4f}  F1: {f1_te:.4f}  P: {p_te:.4f}  R: {r_te:.4f}")
print(f"Gap Acc: {acc_tr - acc_te:.4f}")
print()
print(classification_report(y_test, y_pred_te, target_names=['Fake', 'Real']))

all_results["TF-IDF + XGBoost"] = {
    "test_acc": round(acc_te, 4),
    "test_f1": round(f1_te, 4),
    "test_precision": round(p_te, 4),
    "test_recall": round(r_te, 4),
    "cm": confusion_matrix(y_test, y_pred_te).tolist(),
}

Features : 5004 (TF-IDF: 5000 + meta: 4)
[XGBoost enrichi] Entrainement (train+valid)...
=== TF-IDF + XGBoost (enrichi) ===
Train -> Acc: 0.7875  F1: 0.7858
Test  -> Acc: 0.6906  F1: 0.6883  P: 0.6885  R: 0.6906
Gap Acc: 0.0969

              precision    recall  f1-score   support

        Fake       0.66      0.60      0.63       553
        Real       0.71      0.76      0.74       714

    accuracy                           0.69      1267
   macro avg       0.69      0.68      0.68      1267
weighted avg       0.69      0.69      0.69      1267



## 4. Entrainement avec SMOTE

**SMOTE** (Synthetic Minority Over-sampling Technique) genere des exemples synthetiques pour la classe minoritaire. On l'applique **apres** la vectorisation TF-IDF et **avant** le classifieur.

**Attention** : SMOTE necessite des matrices denses ou accepte sparse. On utilise `imblearn.pipeline.Pipeline` qui gere correctement le resampling dans la cross-validation.

In [22]:
# SMOTE + LogisticRegression
best_tfidf_params = grid_lr.best_params_

tfidf_smote = TfidfVectorizer(
    sublinear_tf=True,
    ngram_range=best_tfidf_params.get("tfidf__ngram_range", (1, 2)),
    min_df=best_tfidf_params.get("tfidf__min_df", 2),
    max_df=best_tfidf_params.get("tfidf__max_df", 0.9),
)

X_train_tfidf = tfidf_smote.fit_transform(X_train)
X_test_tfidf = tfidf_smote.transform(X_test)

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_tfidf, y_train)

print(f"Avant SMOTE : {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Apres SMOTE : {dict(zip(*np.unique(y_train_sm, return_counts=True)))}")

lr_smote = LogisticRegression(max_iter=1000, random_state=42, C=best_tfidf_params.get("clf__C", 1.0))
lr_smote.fit(X_train_sm, y_train_sm)

y_pred_smote = lr_smote.predict(X_test_tfidf)
acc = accuracy_score(y_test, y_pred_smote)
f1 = f1_score(y_test, y_pred_smote, average="weighted")

all_results["TF-IDF + SMOTE + LogReg"] = {
    "test_acc": round(acc, 4), "test_f1": round(f1, 4),
    "test_precision": round(precision_score(y_test, y_pred_smote, average="weighted"), 4),
    "test_recall": round(recall_score(y_test, y_pred_smote, average="weighted"), 4),
    "cm": confusion_matrix(y_test, y_pred_smote).tolist(),
}
print(f"\n=== TF-IDF + SMOTE + LogReg ===")
print(f"Test — Acc: {acc:.4f}  F1: {f1:.4f}")
print(classification_report(y_test, y_pred_smote, target_names=["Fake", "Real"]))

Avant SMOTE : {np.int64(0): np.int64(4488), np.int64(1): np.int64(5752)}
Apres SMOTE : {np.int64(0): np.int64(5752), np.int64(1): np.int64(5752)}

=== TF-IDF + SMOTE + LogReg ===
Test — Acc: 0.6148  F1: 0.6163
              precision    recall  f1-score   support

        Fake       0.55      0.62      0.58       553
        Real       0.67      0.61      0.64       714

    accuracy                           0.61      1267
   macro avg       0.61      0.62      0.61      1267
weighted avg       0.62      0.61      0.62      1267



In [23]:
# SMOTE + LinearSVC
svc_smote = LinearSVC(max_iter=2000, random_state=42, C=grid_svc.best_params_.get("clf__C", 1.0))
svc_smote.fit(X_train_sm, y_train_sm)

y_pred_svc_sm = svc_smote.predict(X_test_tfidf)
all_results["TF-IDF + SMOTE + LinearSVC"] = {
    "test_acc": round(accuracy_score(y_test, y_pred_svc_sm), 4),
    "test_f1": round(f1_score(y_test, y_pred_svc_sm, average="weighted"), 4),
    "test_precision": round(precision_score(y_test, y_pred_svc_sm, average="weighted"), 4),
    "test_recall": round(recall_score(y_test, y_pred_svc_sm, average="weighted"), 4),
    "cm": confusion_matrix(y_test, y_pred_svc_sm).tolist(),
}
print(f"=== TF-IDF + SMOTE + LinearSVC ===")
print(f"Test — Acc: {accuracy_score(y_test, y_pred_svc_sm):.4f}  F1: {f1_score(y_test, y_pred_svc_sm, average='weighted'):.4f}")
print(classification_report(y_test, y_pred_svc_sm, target_names=["Fake", "Real"]))

=== TF-IDF + SMOTE + LinearSVC ===
Test — Acc: 0.6275  F1: 0.6285
              precision    recall  f1-score   support

        Fake       0.57      0.60      0.59       553
        Real       0.68      0.65      0.66       714

    accuracy                           0.63      1267
   macro avg       0.62      0.62      0.62      1267
weighted avg       0.63      0.63      0.63      1267



## 5. Word Embeddings

### 5.1 Word2Vec entraine sur le corpus LIAR

On entraine Word2Vec (Skip-gram) directement sur les textes du dataset pour capturer le vocabulaire politique specifique.

In [24]:
from gensim.models import Word2Vec

sentences_train = [text.split() for text in X_train]

w2v_model = Word2Vec(
    sentences_train, vector_size=100, window=5,
    min_count=2, workers=4, epochs=20, sg=1, seed=42,
)
print(f"Word2Vec entraine : {len(w2v_model.wv)} mots, dim {w2v_model.vector_size}")

def text_to_vec(text, model, dim=100):
    """Moyenne des vecteurs Word2Vec pour un texte."""
    words = text.split()
    vecs = [model.wv[w] for w in words if w in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(dim)

X_train_w2v = np.vstack([text_to_vec(t, w2v_model) for t in X_train])
X_test_w2v = np.vstack([text_to_vec(t, w2v_model) for t in X_test])

lr_w2v = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
lr_w2v.fit(X_train_w2v, y_train)
y_pred_w2v = lr_w2v.predict(X_test_w2v)

all_results["Word2Vec (corpus) + LogReg"] = {
    "test_acc": round(accuracy_score(y_test, y_pred_w2v), 4),
    "test_f1": round(f1_score(y_test, y_pred_w2v, average="weighted"), 4),
    "test_precision": round(precision_score(y_test, y_pred_w2v, average="weighted"), 4),
    "test_recall": round(recall_score(y_test, y_pred_w2v, average="weighted"), 4),
    "cm": confusion_matrix(y_test, y_pred_w2v).tolist(),
}
print(f"\n=== Word2Vec (corpus) + LogReg ===")
print(f"Test — Acc: {accuracy_score(y_test, y_pred_w2v):.4f}  F1: {f1_score(y_test, y_pred_w2v, average='weighted'):.4f}")
print(classification_report(y_test, y_pred_w2v, target_names=["Fake", "Real"]))

Word2Vec entraine : 6620 mots, dim 100

=== Word2Vec (corpus) + LogReg ===
Test — Acc: 0.5943  F1: 0.5960
              precision    recall  f1-score   support

        Fake       0.53      0.61      0.57       553
        Real       0.66      0.59      0.62       714

    accuracy                           0.59      1267
   macro avg       0.59      0.60      0.59      1267
weighted avg       0.60      0.59      0.60      1267



### 5.2 GloVe pre-entraine + Feature Engineering avec metadonnees

On compare avec GloVe (pre-entraine sur Wikipedia/Gigaword) et on teste l'ajout de features metadonnees (encodage du parti politique) combinees aux embeddings.

In [25]:
import gensim.downloader as api

print("Chargement GloVe (glove-wiki-gigaword-100)...")
glove_model = api.load("glove-wiki-gigaword-100")
print(f"GloVe charge : {len(glove_model)} mots, dim {glove_model.vector_size}")

def text_to_glove(text, model, dim=100):
    words = str(text).lower().split()
    vecs = [model[w] for w in words if w in model]
    return np.mean(vecs, axis=0) if vecs else np.zeros(dim)

X_train_glove = np.vstack(X_train.apply(lambda t: text_to_glove(t, glove_model)))
X_test_glove = np.vstack(X_test.apply(lambda t: text_to_glove(t, glove_model)))

# GloVe seul
lr_glove = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
lr_glove.fit(X_train_glove, y_train)
y_pred_glove = lr_glove.predict(X_test_glove)

all_results["GloVe + LogReg"] = {
    "test_acc": round(accuracy_score(y_test, y_pred_glove), 4),
    "test_f1": round(f1_score(y_test, y_pred_glove, average="weighted"), 4),
    "test_precision": round(precision_score(y_test, y_pred_glove, average="weighted"), 4),
    "test_recall": round(recall_score(y_test, y_pred_glove, average="weighted"), 4),
    "cm": confusion_matrix(y_test, y_pred_glove).tolist(),
}
print(f"=== GloVe + LogReg ===")
print(f"Test — Acc: {accuracy_score(y_test, y_pred_glove):.4f}  F1: {f1_score(y_test, y_pred_glove, average='weighted'):.4f}")

# GloVe + metadonnees (party encoding)
# On utilise un mapping manuel pour gerer les valeurs inconnues au test
known_parties = df_train["party"].fillna("unknown").unique()
party_to_int = {p: i for i, p in enumerate(known_parties)}

def encode_party(series, mapping):
    """Encode le parti politique — les partis inconnus recoivent la valeur -1."""
    return series.fillna("unknown").map(lambda p: mapping.get(p, -1)).values

party_train = encode_party(df_train["party"], party_to_int)
party_test = encode_party(df_test["party"], party_to_int)

print(f"\nPartis connus (train) : {len(known_parties)}")
n_unknown = (party_test == -1).sum()
print(f"Partis inconnus dans test : {n_unknown} ({n_unknown/len(party_test)*100:.1f}%)")

X_train_meta = np.column_stack([X_train_glove, party_train])
X_test_meta = np.column_stack([X_test_glove, party_test])

lr_meta = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
lr_meta.fit(X_train_meta, y_train)
y_pred_meta = lr_meta.predict(X_test_meta)

all_results["GloVe + Party + LogReg"] = {
    "test_acc": round(accuracy_score(y_test, y_pred_meta), 4),
    "test_f1": round(f1_score(y_test, y_pred_meta, average="weighted"), 4),
    "test_precision": round(precision_score(y_test, y_pred_meta, average="weighted"), 4),
    "test_recall": round(recall_score(y_test, y_pred_meta, average="weighted"), 4),
    "cm": confusion_matrix(y_test, y_pred_meta).tolist(),
}
print(f"\n=== GloVe + Party (metadata) + LogReg ===")
print(f"Test — Acc: {accuracy_score(y_test, y_pred_meta):.4f}  F1: {f1_score(y_test, y_pred_meta, average='weighted'):.4f}")
print(f"\nNote : L'ajout de metadonnees (party) peut ameliorer ou degrader les resultats.")
print("C'est un signal de biais potentiel si le party encoding ameliore fortement les scores.")

Chargement GloVe (glove-wiki-gigaword-100)...
GloVe charge : 400000 mots, dim 100
=== GloVe + LogReg ===
Test — Acc: 0.5872  F1: 0.5886

Partis connus (train) : 24
Partis inconnus dans test : 1 (0.1%)

=== GloVe + Party (metadata) + LogReg ===
Test — Acc: 0.5919  F1: 0.5933

Note : L'ajout de metadonnees (party) peut ameliorer ou degrader les resultats.
C'est un signal de biais potentiel si le party encoding ameliore fortement les scores.


## 6. Comparaison des modeles et visualisations

In [32]:
# Tableau comparatif
results_df = pd.DataFrame(all_results).T
results_df = results_df[["test_acc", "test_f1", "test_precision", "test_recall"]].copy()
results_df.columns = ["Accuracy", "F1-Score", "Precision", "Recall"]
results_df = results_df.sort_values("F1-Score", ascending=False)
print("=== Comparaison de tous les modeles ===")
results_df

=== Comparaison de tous les modeles ===


,Accuracy,F1-Score,Precision,Recall
TF-IDF + XGBoost,0.6906,0.6883,0.6885,0.6906
TF-IDF + LinearSVC,0.6401,0.6407,0.6417,0.6401
TF-IDF + SMOTE + LinearSVC,0.6275,0.6285,0.6304,0.6275
TF-IDF + SMOTE + LogReg,0.6148,0.6163,0.621,0.6148
TF-IDF + LogReg,0.6006,0.6022,0.6094,0.6006
Word2Vec (corpus) + LogReg,0.5943,0.596,0.6021,0.5943
GloVe + Party + LogReg,0.5919,0.5933,0.5959,0.5919
GloVe + LogReg,0.5872,0.5886,0.5918,0.5872


In [27]:
# Graphique comparatif
fig = go.Figure()
models = results_df.index.tolist()
fig.add_trace(go.Bar(name="Accuracy", x=models, y=results_df["Accuracy"], marker_color="#3498db"))
fig.add_trace(go.Bar(name="F1-Score", x=models, y=results_df["F1-Score"], marker_color="#e74c3c"))
fig.update_layout(
    barmode="group", title="Comparaison des modeles de base — Accuracy vs F1-Score",
    yaxis_title="Score", template="plotly_white", height=500,
    yaxis=dict(range=[0, 1]),
)
fig.show()

In [28]:
# Matrices de confusion
fig = make_subplots(rows=1, cols=2, subplot_titles=["TF-IDF + LogReg", "TF-IDF + SMOTE + LogReg"])
for i, name in enumerate(["TF-IDF + LogReg", "TF-IDF + SMOTE + LogReg"]):
    cm = np.array(all_results[name]["cm"])
    fig.add_trace(go.Heatmap(
        z=cm, x=["Fake", "Real"], y=["Fake", "Real"],
        colorscale="Blues", showscale=False,
        text=cm, texttemplate="%{text}", textfont={"size": 16},
    ), row=1, col=i+1)
fig.update_yaxes(title_text="Reel", row=1, col=1)
fig.update_xaxes(title_text="Predit")
fig.update_layout(title="Matrices de confusion (sans/avec SMOTE)", template="plotly_white", height=400)
fig.show()

## 7. Sauvegarde des modeles et metriques

In [30]:
# Sauvegarde des pipelines complets
joblib.dump(best_lr, MODELS_DIR / "tfidf_logreg_binary.joblib")
joblib.dump(best_svc, MODELS_DIR / "tfidf_linearsvc_binary.joblib")
joblib.dump(best_xgb, MODELS_DIR / "tfidf_xgboost_binary.joblib")

# SMOTE models
joblib.dump(tfidf_smote, MODELS_DIR / "tfidf_vectorizer_smote.joblib")
joblib.dump(lr_smote, MODELS_DIR / "logreg_smote_binary.joblib")
joblib.dump(svc_smote, MODELS_DIR / "linearsvc_smote_binary.joblib")

# Embeddings
joblib.dump(lr_glove, MODELS_DIR / "glove_logreg_binary.joblib")
joblib.dump(lr_w2v, MODELS_DIR / "w2v_logreg_binary.joblib")
w2v_model.save(str(MODELS_DIR / "w2v_liar.model"))

# Metriques
with open(MODELS_DIR / "baselines_metrics.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, default=str)

print("Modeles et metriques sauvegardes dans", MODELS_DIR)
print(f"XGBoost sauvegarde -> {MODELS_DIR / 'tfidf_xgboost_binary.joblib'}")

NameError: name 'best_xgb' is not defined

## 8. Synthese

**Resultats attendus** :
- TF-IDF + LogReg/LinearSVC : ~60-63% accuracy (baseline lineaire solide)
- TF-IDF + XGBoost : potentiellement superieur grace aux interactions non lineaires
- SMOTE : ameliore le recall de la classe minoritaire, parfois au detriment de la precision
- Word2Vec (corpus) : capture le vocabulaire specifique mais petit corpus → resultats variables
- GloVe (pre-entraine) : bonne couverture mais pas adapte au domaine politique
- GloVe + Party : l'ajout de metadonnees peut ameliorer mais introduit un biais (speaker leakage)

**Observations** :
- Le LIAR dataset est intrinsequement difficile (meme les humains ne sont pas parfaits)
- TF-IDF avec bigrams capture des patterns lexicaux utiles ("no evidence", "not true")
- XGBoost apporte une capacite de modelisation non lineaire sur les features TF-IDF
- Le gap de generalisation (train vs test) mesure l'overfitting — XGBoost devrait mieux le controler
- L'ajout de `party` comme feature est un signal de biais : le modele pourrait apprendre "republican → fake"

**Prochaine etape** : Modeles avances (DistilBERT, RoBERTa, DistilBERT+XGBoost) pour capturer le contexte semantique.